<h2>Impact (v0.64) - 'Impact (v0.64).ipynb'</h2>

<u><h4>Context</h4></u>

The aim of the 'Impact' notebooks are to explore economic impacts resulsting from the FEM outputs in the lead-up to Autumn 2026, taking foward outputs from the associated 'Trade' notebooks.

<u><h4>Version Documentation and Future Work</h4></u>

'v0.2' cleaned up the analysis code by introducing the cost intensity estimation functions that were previously in the 'inventories' notebooks. For example, the 'get_usd_per_tnm' function is included here and applied to voyages identified as relevant to each individual trade record. The NZF costs associated with each trade record are now evaluated like this. 'v0.3' introduced the exporter- and importer-side trade portfolios at an HS-digit 4 level of resolution. With this implemented, the matching algorithm is now be being evaluated using the cargo weight field rather than value, and the process will be being conducted at the HS-digit 4 level of resolution. Since we are now confident regarding the use of the Global Shipping Watch dataset for the purposes of matching HS codes with facilitating vessel types, we should then be in good shape to take the analysis forward using the results produced in this notebook. 'v0.3' also subsumed the 'inventories' notebooks. In 'v0.4' sought to include transport costs data from UNCTAD's Trade-and-Transport database. However, there are now concerns regarding this dataset as it is importer-side only, and therefore has odd looking exporter-side portfolios. Instead, it makes sense for us to utilise trade portfolios derived from UN Comtrade, where these are submitted with disaggregation by Transport Mode. Looking at the 16 countries of interest across African and Caribbean economies we're interested in the summer of 2026, 13 of these have a trade portfolio with MoT Breakdown available across at least one year within the last decade. Therefore this approach makes more sense, and 'v0.6' was amended to work with Comtrade import and export trade portfolios downloaded from the platform. 'v0.61' made use of the updated 'trade' notebooks that derive a new weight field using Value-to-Weight and Weight-to-Value ratios observed in the CEPII BACI datasets across 2017-23. In the case of Trinidad and Tobaogo, import volumes were around 65% higher under the WtV approach which corresponded to a 39% greater 2041 BAU impact from FEM costs, but this was slightly less for exports where the weighted WtV was 9% more but 2041 FEM costs under BAU were around 10% less. A new 'composite' weight field has been added in Trade (v0.61) which only introduces the VtW-derived values where 'netWgt' is observed to be zero, however this resulted in only a tiny increase in recorded weight of around 1% for Angola, whereas the full VtW-derived field was ~50% greater. 

'v0.62' makes use of a new 'transport cost' field derived using ad-valorem transport cost rates identified from UNCTAD's Trade-and-Transport database. With these, we can now introduce another results disaggregation that presents NZF costs relative to expected transport costs, which is then fully in-line with how impacts are expressed in UNCTAD's CIA Task 3 work. 'v0.63' includes an assessment of 'Cost Intensity' matching process (similar to CIF-FOB matching), plus metrics on 'fac_vessel_types' and 'average sea-distance' (introduced in Trade v0.63). 'v0.64' has been modified throughout to account for a new scenario, namely the 'Japan' proposal, plus to remove the 'Low-then-High' scenario. In terms of the Impact notebook, this remains the same as 'v0.63'.

In [1]:
# Import packages
import os
import ast
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def voyages_analysis(voys):
    # Define a function to analyses key statistics from a given set of voyages, 'voys'.
    print("Number of Voyages found: {0}".format(voys.shape[0]))
    print("Total size of the fleet: {0} vessels".format(len(voys.imo.unique())))
    print("{0} vessel types contained in the dataset: {1}".format(len(voys.vessel_type.unique()), voys.vessel_type.unique()))
    print("Total transport work capacity: {0} million deadweight-tonne nautical miles".format(np.round(voys.dwtnm.sum() / 1e6, 1)))
    print("Total cargo transported: {0} tonnes".format(np.round(voys.tonnes.sum(), 1)))
    print("Total transport work: {0} million tonne nautical miles".format(np.round(voys.tnm.sum() / 1e6, 1)))
    print("Average build year of the fleet: {0}".format(int(np.round(voys.year_of_built.mean(), 0))))
    print("Total Energy Demands: {0} EJ".format(np.round(voys.energy_tj.sum(), 1)))
    print("Total CO2 produced: {0} tonnes".format(np.round(voys.co2_t.sum(), 1)))

def get_usd_per_dwtnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carrying capacity) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values * voys.deadweight.values).sum() / voys.deadweight.sum()
    
def get_usd_per_tnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carried) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values / voys.class_util.values * voys.tonnes.values).sum() / voys.tonnes.sum()

def download_link(df, title = "Download CSV file", filename = "data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

version = "v0.64"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
voys_fem_dir = "/Users/apple/repos/datasets/ucl/fleet evolution model/2026 annex vi amendments (v0.2)/2. NZF Impact Costs for 2018 IMO IV Voyages/voys_fem_v0.2.csv"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/impact/"
res_mlc = True

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "388"
iso_country = "Jamaica"
year = 2024
gdp = 22.010e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."
os.makedirs(output_dir+"{0}/results/".format(iso_country)) if not os.path.exists(output_dir+"{0}/results/".format(iso_country)) else "Output results folder already exists."

## 1.2 Voyages with FEM Results

In [3]:
voys_fem_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}

voys_fem = pd.read_csv(voys_fem_dir, dtype=voys_fem_d)
voys_fem["dwtnm"] = voys_fem.deadweight * voys_fem.dist_travelled_ais
voys_fem["tonnes"] = voys_fem.deadweight * voys_fem.class_util
voys_fem["tnm"] = voys_fem.tonnes * voys_fem.dist_travelled_ais
voys_fem.loc[voys_fem.source_country == "Namibia", "source_iso_2"] = "NA"
voys_fem.loc[voys_fem.dest_country == "Namibia", "dest_iso_2"] = "NA"

countries_set = pd.Series(list(set(list(voys_fem.source_country.unique()) + list(voys_fem.dest_country.unique())))).sort_values()
print("Number of countries present in dataset: {0}.".format(len(countries_set)))
print("Number of IMO numbers present in dataset: {0}.".format(len(voys_fem.imo.unique())))
print("Total CO2 contained within the dataset: {0} million tonnes.".format(np.round(voys_fem.co2_t.sum() / 1e6, 1)))
print("Vessel types present in dataset: \n\n{0}\n".format(pd.Series(voys_fem.vessel_type.unique()).sort_values().values))
#print("\nCountries in dataset: \n\n", countries_set.values, "\n")
voys_fem.head(2)

Number of countries present in dataset: 207.
Number of IMO numbers present in dataset: 26132.
Total CO2 contained within the dataset: 544.7 million tonnes.
Vessel types present in dataset: 

['Bulk carrier' 'Chemical tanker' 'Container' 'General cargo'
 'Liquefied gas tanker' 'Oil tanker' 'Refrigerated bulk' 'Vehicle']



,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,0,5034525,413781699,CNTAX,156,CHN,CN,China,Taixing,CNNKG,156,CHN,CN,China,Nanjing,2018-01-01 06:00:00,2018-01-01 19:00:00,13.0,1.0,14.0,62.568523,0.0,0.0,0.0,3.635661,0.338483,3.974144,0.0,0.0,0.0,0.0,0.0,0.0,0.155243,0.014453,11.655930,1.085176,12.741107,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,0.169696,Refrigerated bulk,3786.0,dwt,3881.0,NaN,NaN,NaN,NaN,2,2000-5999,1959.0,Panama,SSD,4,76.1,0.666667,0.03155,0.032676,0.032143,0.031676,0.031218,0.030769,0.03031,0.029878,0.029438,0.031576,0.052178,0.052414,0.05804,0.060001,0.059297,0.058435,0.058228,0.058319,0.031568,0.034644,0.036843,0.044726,0.049621,0.053882,0.057797,0.059065,0.057999,0.031568,0.032727,0.034529,0.040735,0.048178,0.055864,0.057447,0.058672,0.05942,0.031559,0.032699,0.033808,0.037977,0.040129,0.041837,0.043315,0.044697,0.045876,0.031559,0.033985,0.034896,0.036131,0.038297,0.039924,0.040606,0.041099,0.041463,242828.439338,7661.320668,7934.549945,7805.304846,7691.950531,7580.696290,7471.542086,7360.199865,7255.307537,7148.350725,7667.458038,12670.391275,12727.642802,14093.647297,14570.027623,14398.931835,14189.590704,14139.488873,14161.462401,7665.715352,8412.516422,8946.6

# 2. Let's consider Imports

## 2.1 Read-in Trade dataset

In [4]:
trade_im_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str", "HS4":"str"}
if res_mlc == False:
    trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_dist_{1}.csv".format(iso_country, version), dtype=trade_im_d)
else:
    trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_mlc_{1}.csv".format(iso_country, version), dtype=trade_im_d)
trade_im['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_im.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_im.sea_dist_nm.min(), 1),
    round(trade_im.sea_dist_nm.max(), 1),
    round(trade_im.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_im.vol_kg_vtw / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_im.cifvalue / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.cifvalue.sum() / 1000), 1)
))
trade_im.head(2)

Aggregate trade portfolio statistics for 'Jamaica': 

Number of trade records present in dataset: 24765
Number of partner countries present in dataset: 116
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$6858.2m
Total weight within the dataset: 6845.2 million kilograms

Distance to market: 263.5 nm (min) to 12646.8 nm (max), 5026.5 nm (observed mean)
Average distance to market (volume-weighted): 2750.9 nm
Average distance to market (value-weighted): 4047.9 nm


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue
0,0,388,Jamaica,276,Germany,000,World,Import,Sea,"220890 | Spirits, liqueurs an",220890,l,2322.0,2322.0,6854.60,NaN,6854.60,"Germany | 220890 | Spirits, liqueurs an",22,"22 | Beverages, spirits a","Germany | 22 | Beverages, spirits a",2208,"2208 | Ethyl alcohol, unden","Germany | 2208 | Ethyl alcohol, unden",22.0,2208.0,220890.0,Undenatured ethyl alcohol of an alcoholic stre...,Food & Beverages,1551.0,"Distilling, rectifying and blending of spirits...",11249.0,"Spirits & distilled alcoholic beverages, n.e.s.",26.0,B_T - Beverages and tobacco products,2413.0,Undenatured ethyl alcohol of an alcoholic stre...,6.3251,0.1581,both,0.338751,1083.712260,2322.0,"[Container, Chemical tanker]","['Container', 'Chemical tanker']",6530.0,4057.552630,4397.219531,"1 - Reporter, Partner and HS4 code",4.79,1,313.33,6854.60,6541.27
1,1,388,Jamaica,124,Canada,000,World,Import,Sea,"220429 | Wine; still, in cont",220429,l,18.5,205.5,548.02,NaN,548.02,"Canada | 220429 | Wine; still, in cont",22,"22 | Beverages, spirits a","Canada | 22 | Beverages, spirits a",2204,2204 | Wine of fresh grapes,Canada | 2204 | Wine of fresh grapes,22.0,2204.0,220429.0,Wine other than sparkling wine of fresh grapes...,Food & Beverages,1552.0,Manufacture of wines,11217.0,Wine of fresh grapes (other than sparkling win...,26.0,B_T - Beverages and tobacco products,24212.0,"Wine of fresh grapes, except sparkling wine; g...",3.4114,0.2931,both,0.374986,160.624662,205.5,[Container],['Container'],4011.0,2492.319081,400.327910,"1 - Reporter, Partner and HS4 code",12.31,1,60.07,548.02,487.95


## 2.2 Derive Internationally Arriving Voyages

In [5]:
# International Arrivals
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_arr_voys)
int_arr_voys.head(2)

Number of Voyages found: 2198
Total size of the fleet: 757 vessels
8 vessel types contained in the dataset: ['Bulk carrier' 'General cargo' 'Chemical tanker' 'Refrigerated bulk'
 'Vehicle' 'Container' 'Liquefied gas tanker' 'Oil tanker']
Total transport work capacity: 147495.7 million deadweight-tonne nautical miles
Total cargo transported: 46008452.6 tonnes
Total transport work: 85425.7 million tonne nautical miles
Average build year of the fleet: 2007
Total Energy Demands: 15374.3 EJ
Total CO2 produced: 1183883.2 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,13146,7915412,308299000,USPCV,840,USA,US,United States of America,Cape Canaveral,JMKIN,388,JAM,JM,Jamaica,Kingston,2018-01-27 01:00:00,2018-01-30 23:00:00,94.0,4.0,98.0,861.039044,52.004859,0.713617,52.718476,3.277137,0.0,3.277137,0.0,0.0,0.0,0.0,0.0,0.0,2.230529,0.028687,172.449634,2.222203,174.671837,019,Americas,021,Northern America,high_income,0,0,019,Americas,419,Latin America and the Caribbean,upp_mid_income,0,1,2.259217,Bulk carrier,46409.0,dwt,74423.0,NaN,NaN,NaN,NaN,4,60000-99999,1981.0,Bahamas,SSD,1,4.1,0.526316,0.002221,0.0023,0.002262,0.00223,0.002198,0.002167,0.002135,0.002105,0.002074,0.002217,0.003631,0.003523,0.003915,0.004274,0.004276,0.004123,0.003866,0.003574,0.002217,0.002414,0.002495,0.003009,0.003449,0.004105,0.003936,0.003844,0.003749,0.002217,0.002282,0.002388,0.002784,0.003264,0.004337,0.004165,0.004176,0.004096,0.002219,0.00229,0.002358,0.002635,0.002787,0.003024,0.003409,0.003302,0.003461,0.002219,0.002379,0.002433,0.002509,0.002646,0.002745,0.00278,0.002801,0.002813,6.408111e+07,142333.084731,147359.746102,144981.994513,142896.589313,140849.819643,138841.684829,136793.296564,134863.568469,132895.859576,142059.137560,232697.724335,225728.406734,250909.110052

## 2.3 Evaluate NZF Costs

In [6]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
vess_types_not_found_in_inventories, vess_types_not_found_in_global_voys, hs_codes_not_mapped_to_HS, cost_int_match_desc, cost_int_match_desc_bool = [], [], [], [], False

for year in years:

    print("Processing game year 20{0}".format(year))

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, asis_usd, brazil_usd, liscr_usd, direct_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_im.iterrows():

        # initiate selected voyages table as empty dataframe to begin
        int_arr_voys_vess = pd.DataFrame()

        # 3. If no facilitating vessel types were picked up for the HS code, take from full primary int. inv. voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.HS4_desc)
            int_arr_voys_vess = int_arr_voys
            cost_int_match_desc.append("3. Primary Int. Inv. without Vess Type") if cost_int_match_desc_bool == False else None

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_arr_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]    
            
            if int_arr_voys_vess.shape[0] > 0:
                cost_int_match_desc.append("1. Primary Int. Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                
            else:
                # 2. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
                int_arr_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]

                if int_arr_voys_vess.shape[0] > 0:
                    cost_int_match_desc.append("2. Global Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

                else:
                    # 4. Use full global voyage dataset as a back-up
                    int_arr_voys_vess = voys_fem
                    cost_int_match_desc.append("4. Global Inv. without Vess Type.") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_global_voys.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

        # Add cost intensities associated with the subset of voyages by vessel type, assuming 50% of NZF costs are borne by the importer.
        bau_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "bau35_usd_dwtnm_"+year))
        levy_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "levy35_usd_dwtnm_"+year))
        asis_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "asis35_usd_dwtnm_"+year))
        brazil_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "brazil35_usd_dwtnm_"+year))
        liscr_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "liscr35_usd_dwtnm_"+year))
        direct_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "direct35_usd_dwtnm_"+year))
    
    if cost_int_match_desc_bool == False:
        trade_im["cost_int_match_desc"] = cost_int_match_desc
    cost_int_match_desc_bool = True
    
    trade_im["bau_usd_"+year] = bau_usd
    trade_im["levy_usd_"+year] = levy_usd
    trade_im["asis_usd_"+year] = asis_usd
    trade_im["brazil_usd_"+year] = brazil_usd
    trade_im["liscr_usd_"+year] = liscr_usd
    trade_im["direct_usd_"+year] = direct_usd

print("Inventory voyages table used for the following vessel types: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))
print("Global voyages table used for the following vessel types found: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_global_voys).unique()))
print("Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))

print("\nNumber of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_im.sea_dist_nm.min(), 1),
    round(trade_im.sea_dist_nm.max(), 1),
    round(trade_im.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_im.vol_kg_vtw / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_im.cifvalue / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.cifvalue.sum() / 1000), 1)
))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.bau_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_im.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.asis_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.brazil_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.liscr_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Direct: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.direct_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.direct_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))

trade_im_agg = trade_im.groupby(["cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "cifvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_im_agg["vol_kg_vtw_pct"] = round(100 * trade_im_agg.vol_kg_vtw / trade_im_agg.vol_kg_vtw.sum(), 1)
trade_im_agg["cifvalue_pct"] = round(100 * trade_im_agg.cifvalue / trade_im_agg.cifvalue.sum(), 1)
print("Most prominent 'cost intensity match' types...\n\n{0}\n".format(
    trade_im_agg[["cost_int_match_desc", "vol_kg_vtw_pct", "cifvalue_pct"]].head(5)
))
trade_im_agg_2 = trade_im.groupby(["fac_vessel_types_str", "cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "cifvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_im_agg_2["vol_kg_vtw_pct"] = round(100 * trade_im_agg_2.vol_kg_vtw / trade_im_agg_2.vol_kg_vtw.sum(), 1)
trade_im_agg_2["cifvalue_pct"] = round(100 * trade_im_agg_2.cifvalue / trade_im_agg_2.cifvalue.sum(), 1)
print("Most prominent 'facilitating vessel types' and their 'cost intensity match' types...")
display(trade_im_agg_2[["fac_vessel_types_str", "cost_int_match_desc", "vol_kg_vtw_pct", "cifvalue_pct"]].head(5))

trade_im_na = trade_im[(trade_im[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_im.fac_vessel_types.str.len() == 0)]
trade_im = trade_im[(~trade_im[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_im.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n".format(trade_im_na.primaryValue.sum(), round(100 * trade_im_na.primaryValue.sum() / trade_im.primaryValue.sum(), 1)))

trade_im.to_csv(output_dir + "{0}/impact_im_{1}.csv".format(iso_country, version), index=False)
trade_im.head(2)

Processing game year 2026
Processing game year 2029
Processing game year 2032
Processing game year 2035
Processing game year 2038
Processing game year 2041
Processing game year 2044
Processing game year 2047
Processing game year 2050
Inventory voyages table used for the following vessel types: 

[]

Global voyages table used for the following vessel types found: 

['Ro-Ro' 'Other liquids tankers' 'Cruise']

Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: 

['2202 | Waters, including mi' '2204 | Wine of fresh grapes'
 '5703 | Carpets and other te' '5704 | Carpets and other te'
 '4412 | Plywood, veneered pa' '4407 | Wood sawn or chipped'
 '4403 | Wood in the rough, w' '4406 | Railway or tramway s'
 '8539 | Lamps; electric fila' '8701 | Tractors; (other tha'
 '8456 | Machine-tools; for w' '8459 | Machine-tools; (incl'
 "4418 | Builders' joinery an" '2106 | Food preparations no'
 '4905 | Maps and hydrographi' '4911 | Printed matter

,fac_vessel_types_str,cost_int_match_desc,vol_kg_vtw_pct,cifvalue_pct
0,['Oil tanker'],1. Primary Int. Inv. with Vess Type,28.7,16.0
1,['Container'],1. Primary Int. Inv. with Vess Type,28.2,46.1
2,['Chemical tanker'],1. Primary Int. Inv. with Vess Type,11.9,7.1
3,['Liquefied gas tanker'],1. Primary Int. Inv. with Vess Type,10.0,5.1
4,['Bulk carrier'],1. Primary Int. Inv. with Vess Type,6.3,2.3


Rows with NA: $0.0 (0.0%)



,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50
0,0,388,Jamaica,276,Germany,000,World,Import,Sea,"220890 | Spirits, liqueurs an",220890,l,2322.0,2322.0,6854.60,NaN,6854.60,"Germany | 220890 | Spirits, liqueurs an",22,"22 | Beverages, spirits a","Germany | 22 | Beverages, spirits a",2208,"2208 | Ethyl alcohol, unden","Germany | 2208 | Ethyl alcohol, unden",22.0,2208.0,220890.0,Undenatured ethyl alcohol of an alcoholic stre...,Food & Beverages,1551.0,"Distilling, rectifying and blending of spirits...",11249.0,"Spirits & distilled alcoholic beverages, n.e.s.",26.0,B_T - Beverages and tobacco products,2413.0,Undenatured ethyl alcohol of an alcoholic stre...,6.3251,0.1581,both,0.338751,1083.712260,2322.0,"[Container, Chemical tanker]","['Container', 'Chemical tanker']",6530.0,4057.552630,4397.219531,"1 - Reporter, Partner and HS4 code",4.79,1,313.33,6854.60,6541.27,1. Primary Int. Inv. with Vess Type,15.568859,15.531179,15.534704,15.534704,15.554305,15.554305,16.297762,29.121174,17.742975,16.559334,16.615276,17.099154,15.952614,28.298233,18.792616,17.247850,16.855513,17.637938,15.649983,31.384326,22.858353,20.421934,18.936004,18.382427,15.352569,33.316038,26.937184,25.162505,20.053986,19.703414,15.060727,33.376920,32.436144,32.489088,21.658855,20.684093,14.763105,33.105293,31.571370,34.119597,26.04316,21.052869,14.482807,29.073400,30.820008,33.073262,24.887277,21.293166,14.197241,26.539420,26.984528,32.246085,26.600109,21.443829
1,1,388,Jamaica,124,Canada,000,World,Import,Sea,"220429 | Wine; still, in cont",220429,l,18.5,205.5,548.02,NaN,548.02,"Canada | 220429 | Wine; still, in cont",22,"22 | Beverages, spirits a","Canada | 22 | Beverages, spirits a",2204,2204 | Wine of fresh grapes,Canada | 2204 | Wine of fresh grapes,22.0,2204.0,220429.0,Wine other than sparkling wine of fresh grapes...,Food & Beverages,1552.0,Manufacture of wines,11217.0,Wine of fresh grapes (other than sparkling win...,26.0,B_T - Beverages and tobacco products,24212.0,"Wine of fresh grapes, except sparkling wine; g...",3.4114,0.2931,both,0.374986,160.624662,205.5,[Container],['Container'],4011.0,2492.319081,400.327910,"1 - Reporter, Partner and HS4 code",12.31,1,60.07,548.02,487.95,1. Primary Int. Inv. with Vess Type,1.372704,1.368993,1.369338,1.369338,1.371302,1.371302,1.441082,2.646714,1.578973,1.471222,1.476449,1.516801,1.408698,2.573796,1.668375,1.530497,1.492735,1.567561,1.380305,2.848677,2.065276,1.820776,1.679716,1.637672,1.352395,3.002950,2.424240,2.258822,1.786545,1.761655,1.325007,3.050322,2.962635,2.943695,1.938527,1.853917,1.297077,3.017117,2.862245,3.122208,2.35932,1.888698,1.270775,2.635934,2.815291,3.003581,2.234244,1.911332,1.243983,2.395233,2.433570,2.930772,2.410196,1.925451


## 2.4 Create Results Tables

### 2.4.1 Total Costs per Scenario

In [7]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_im["bau_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["levy_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["asis_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["brazil_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["liscr_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["direct_usd_{0}".format(year)].sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% cif)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% fob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_fob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["direct_usd_{0}".format(year)].sum() / gdp, 3),
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,7.000454e+07,6.864961e+07,6.751850e+07,6.641243e+07,6.530854e+07,6.415505e+07,6.307812e+07,6.190415e+07
1,Levy,1.187500e+08,1.155952e+08,1.272610e+08,1.380013e+08,1.398911e+08,1.343417e+08,1.207551e+08,1.107325e+08
2,'as-is',7.488996e+07,7.871995e+07,9.436238e+07,1.108366e+08,1.322612e+08,1.298337e+08,1.254074e+08,1.137919e+08
3,Brazil,7.069911e+07,7.331840e+07,8.561322e+07,1.039985e+08,1.351342e+08,1.400772e+08,1.359784e+08,1.319121e+08
4,LISCR,7.095369e+07,7.168447e+07,7.967407e+07,8.379098e+07,8.980818e+07,1.089848e+08,1.057525e+08,1.091106e+08
5,Direct,7.296780e+07,7.485649e+07,7.783844e+07,8.307796e+07,8.705030e+07,8.850388e+07,8.989663e+07,9.063301e+07



Total FEM costs by scenario (% ciffob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,11.25,11.03,10.85,10.67,10.50,10.31,10.14,9.95
1,Levy,19.09,18.58,20.45,22.18,22.49,21.59,19.41,17.80
2,'as-is',12.04,12.65,15.17,17.82,21.26,20.87,20.16,18.29
3,Brazil,11.36,11.78,13.76,16.72,21.72,22.51,21.86,21.20
4,LISCR,11.40,11.52,12.81,13.47,14.44,17.52,17.00,17.54
5,Direct,11.73,12.03,12.51,13.35,13.99,14.23,14.45,14.57



Total FEM costs by scenario (% cif)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1.02,1.00,0.98,0.97,0.95,0.94,0.92,0.90
1,Levy,1.73,1.69,1.86,2.01,2.04,1.96,1.76,1.61
2,'as-is',1.09,1.15,1.38,1.62,1.93,1.89,1.83,1.66
3,Brazil,1.03,1.07,1.25,1.52,1.97,2.04,1.98,1.92
4,LISCR,1.03,1.05,1.16,1.22,1.31,1.59,1.54,1.59
5,Direct,1.06,1.09,1.13,1.21,1.27,1.29,1.31,1.32



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1.12,1.10,1.08,1.06,1.05,1.03,1.01,0.99
1,Levy,1.90,1.85,2.04,2.21,2.24,2.15,1.94,1.78
2,'as-is',1.20,1.26,1.51,1.78,2.12,2.08,2.01,1.82
3,Brazil,1.13,1.18,1.37,1.67,2.17,2.25,2.18,2.12
4,LISCR,1.14,1.15,1.28,1.34,1.44,1.75,1.70,1.75
5,Direct,1.17,1.20,1.25,1.33,1.40,1.42,1.44,1.45



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.318,0.312,0.307,0.302,0.297,0.291,0.287,0.281
1,Levy,0.540,0.525,0.578,0.627,0.636,0.610,0.549,0.503
2,'as-is',0.340,0.358,0.429,0.504,0.601,0.590,0.570,0.517
3,Brazil,0.321,0.333,0.389,0.473,0.614,0.636,0.618,0.599
4,LISCR,0.322,0.326,0.362,0.381,0.408,0.495,0.480,0.496
5,Direct,0.332,0.340,0.354,0.377,0.396,0.402,0.408,0.412


### 2.4.2 Costs per Scenario relative to BAU

In [8]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_usd_bau_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal change in FEM costs by scenario (% bau)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_bau_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% cif)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% fob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_fob_{1}.csv".format(iso_country, version), index=False)
    display(df)
    
print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,48745506.78,46945552.62,59742479.13,71588867.33,74582570.83,70186649.38,57676966.05,48828335.95
1,'as-is',4885423.09,10070342.69,26843880.19,44424179.19,66952651.04,65678691.65,62329271.07,51887755.34
2,Brazil,694573.08,4668786.58,18094727.03,37586093.22,69825702.62,75922193.68,72900269.90,70007936.08
3,LISCR,949148.29,3034860.28,12155573.61,17378551.53,24499642.25,44829760.57,42674415.12,47206452.37
4,Direct,2963261.22,6206880.62,10319942.61,16665530.39,21741753.96,24348832.96,26818515.48,28728861.29



Total change in FEM costs by scenario (% bau)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,69.63,68.38,88.48,107.79,114.20,109.40,91.44,78.88
1,'as-is',4.23,9.04,15.28,25.09,33.29,37.95,42.52,46.41
2,Brazil,6.98,14.67,39.76,66.89,102.52,102.37,98.81,83.82
3,LISCR,0.99,6.80,26.80,56.59,106.92,118.34,115.57,113.09
4,Direct,1.36,4.42,18.00,26.17,37.51,69.88,67.65,76.26



Total FEM costs by scenario (% ciffob)...



...

,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.71,0.68,0.87,1.04,1.09,1.02,0.84,0.71
1,'as-is',0.04,0.09,0.15,0.24,0.32,0.36,0.39,0.42
2,Brazil,0.07,0.15,0.39,0.65,0.98,0.96,0.91,0.76
3,LISCR,0.01,0.07,0.26,0.55,1.02,1.11,1.06,1.02
4,Direct,0.01,0.04,0.18,0.25,0.36,0.65,0.62,0.69



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.78,0.75,0.96,1.15,1.20,1.13,0.92,0.78
1,'as-is',0.05,0.10,0.17,0.27,0.35,0.39,0.43,0.46
2,Brazil,0.08,0.16,0.43,0.71,1.07,1.05,1.00,0.83
3,LISCR,0.01,0.07,0.29,0.60,1.12,1.22,1.17,1.12
4,Direct,0.02,0.05,0.19,0.28,0.39,0.72,0.68,0.76



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.221,0.213,0.271,0.325,0.339,0.319,0.262,0.222
1,'as-is',0.013,0.028,0.047,0.076,0.099,0.111,0.122,0.131
2,Brazil,0.022,0.046,0.122,0.202,0.304,0.298,0.283,0.236
3,LISCR,0.003,0.021,0.082,0.171,0.317,0.345,0.331,0.318
4,Direct,0.004,0.014,0.055,0.079,0.111,0.204,0.194,0.214


# 3. Let's consider Exports

## 3.1 Read-in Trade dataset

In [9]:
trade_ex_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str", "HS4":"str"}
if res_mlc == False:
    trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_dist_{1}.csv".format(iso_country, version), dtype=trade_ex_d)
else:
    trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_mlc_{1}.csv".format(iso_country, version), dtype=trade_ex_d)
trade_ex['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_ex.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_ex.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_ex.sea_dist_nm.min(), 1),
    round(trade_ex.sea_dist_nm.max(), 1),
    round(trade_ex.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_ex.vol_kg_vtw / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_ex.fobvalue / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.fobvalue.sum() / 1000), 1)
))
trade_ex.head(2)

Aggregate trade portfolio statistics for 'Jamaica': 

Number of trade records present in dataset: 2028
Number of partner countries present in dataset: 70
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1677.9m
Total weight within the dataset: 4602.8 million kilograms

Distance to market: 262.8 nm (min) to 12384.5 nm (max), 1562.5 nm (observed mean)
Average distance to market (volume-weighted): 2467.1 nm
Average distance to market (value-weighted): 2961.2 nm


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue
0,0,388,Jamaica,842,USA,000,World,Domestic Export,Sea,271019 | Petroleum oils and o,271019,kg,5.044147e+08,5.044147e+08,NaN,376375364,376375364,USA | 271019 | Petroleum oils and o,27,"27 | Mineral fuels, miner","USA | 27 | Mineral fuels, miner",2710,2710 | Petroleum oils and o,USA | 2710 | Petroleum oils and o,27.0,2710.0,271019.0,Petroleum oils & oils obtained from bituminous...,"Petroleum, Chemical and Non-Metallic Mineral P...",2320.0,Manufacture of refined petroleum products,3346.0,Petroleum oils & oils obtained from bituminous...,32.0,"P_C - Petroleum, coal products",333.0,Petroleum oils and oils obtained from bitumino...,0.6461,1.5476,both,1.340191,5.824785e+08,5.044147e+08,[Oil tanker],['Oil tanker'],1948.0,1210.430708,7.050499e+08,3 - Reporter and HS4 code,5.47,31,20587732.41,3.969631e+08,376375364
1,1,388,Jamaica,842,USA,000,World,Domestic Export,Sea,330130 | Resinoids,330130,kg,1.707735e+05,1.707735e+05,NaN,599812,599812,USA | 330130 | Resinoids,33,33 | Essential oils and r,USA | 33 | Essential oils and r,3301,3301 | Oils; essential (con,USA | 3301 | Oils; essential (con,33.0,3301.0,330130.0,Resinoids,"Petroleum, Chemical and Non-Metallic Mineral P...",2429.0,Manufacture of other chemical products n.e.c.,55133.0,Resinoids,33.0,"CRP - Chemical, rubber, plastic products",3541.0,"Essential oils and concentrates, aqueous disti...",22.5983,0.0443,both,0.284712,2.657167e+04,1.707735e+05,"[Bulk carrier, Container]","['Bulk carrier', 'Container']",1948.0,1210.430708,3.216317e+04,3 - Reporter and HS4 code,2.78,12,16674.77,6.164868e+05,599812


## 3.2 Derive Internationally Departing Voyages

In [10]:
# International Departures
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_dep_voys)
int_dep_voys.head(2)

Number of Voyages found: 2206
Total size of the fleet: 758 vessels
8 vessel types contained in the dataset: ['Bulk carrier' 'General cargo' 'Chemical tanker' 'Refrigerated bulk'
 'Vehicle' 'Container' 'Liquefied gas tanker' 'Oil tanker']
Total transport work capacity: 147878.0 million deadweight-tonne nautical miles
Total cargo transported: 46138312.1 tonnes
Total transport work: 88561.5 million tonne nautical miles
Average build year of the fleet: 2007
Total Energy Demands: 14091.0 EJ
Total CO2 produced: 1075556.0 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,13147,7915412,308299000,JMKIN,388,JAM,JM,Jamaica,Kingston,COPBO,170,COL,CO,Colombia,Puerto Bolivar,2018-01-31 03:00:00,2018-02-01 22:00:00,43.0,13.0,56.0,458.553760,30.012772,2.575045,32.587817,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.206513,0.103517,93.459772,8.018689,101.478461,019,Americas,419,Latin America and the Caribbean,upp_mid_income,0,1,019,Americas,419,Latin America and the Caribbean,upp_mid_income,0,0,1.310030,Bulk carrier,46409.0,dwt,74423.0,NaN,NaN,NaN,NaN,4,60000-99999,1981.0,Bahamas,SSD,1,4.1,0.526316,0.002221,0.0023,0.002262,0.00223,0.002198,0.002167,0.002135,0.002105,0.002074,0.002217,0.003631,0.003523,0.003915,0.004274,0.004276,0.004123,0.003866,0.003574,0.002217,0.002414,0.002495,0.003009,0.003449,0.004105,0.003936,0.003844,0.003749,0.002217,0.002282,0.002388,0.002784,0.003264,0.004337,0.004165,0.004176,0.004096,0.002219,0.00229,0.002358,0.002635,0.002787,0.003024,0.003409,0.003302,0.003461,0.002219,0.002379,0.002433,0.002509,0.002646,0.002745,0.00278,0.002801,0.002813,3.412695e+07,75800.710282,78477.702093,77211.410003,76100.809501,75010.784683,73941.335190,72850.448376,71822.755056,70774.834736,75654.817356,123925.177473,120213.607353,133623.807803,145872.749456,145

## 3.3 Evaluate NZF Costs

In [11]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
vess_types_not_found_in_inventories, vess_types_not_found_in_global_voys, hs_codes_not_mapped_to_HS, cost_int_match_desc, cost_int_match_desc_bool = [], [], [], [], False

for year in years:

    print("Processing game year 20{0}".format(year))

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, asis_usd, brazil_usd, liscr_usd, direct_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_ex.iterrows():

        # initiate selected voyages table as empty dataframe to begin
        int_dep_voys_vess = pd.DataFrame()

        # 3. If no facilitating vessel types were picked up for the HS code, take from full primary int. inv. voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.HS4_desc)
            int_dep_voys_vess = int_dep_voys
            cost_int_match_desc.append("3. Primary Int. Inv. without Vess Type") if cost_int_match_desc_bool == False else None

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_dep_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]    
            
            if int_dep_voys_vess.shape[0] > 0:
                cost_int_match_desc.append("1. Primary Int. Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                
            else:
                # 2. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
                int_dep_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]
                
                if int_dep_voys_vess.shape[0] > 0:
                    cost_int_match_desc.append("2. Global Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

                else:
                    # 4. Use full global voyage dataset as a back-up
                    int_dep_voys_vess = voys_fem
                    cost_int_match_desc.append("4. Global Inv. without Vess Type.") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_global_voys.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
    
        # Add cost intensities associated with the subset of voyages by vessel type, assuming 50% of NZF costs are borne by the exporter.
        bau_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "bau35_usd_dwtnm_"+year))
        levy_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "levy35_usd_dwtnm_"+year))
        asis_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "asis35_usd_dwtnm_"+year))
        brazil_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "brazil35_usd_dwtnm_"+year))
        liscr_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "liscr35_usd_dwtnm_"+year))
        direct_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "direct35_usd_dwtnm_"+year))

    if cost_int_match_desc_bool == False:
        trade_ex["cost_int_match_desc"] = cost_int_match_desc
    cost_int_match_desc_bool = True

    trade_ex["bau_usd_"+year] = bau_usd
    trade_ex["levy_usd_"+year] = levy_usd
    trade_ex["asis_usd_"+year] = asis_usd
    trade_ex["brazil_usd_"+year] = brazil_usd
    trade_ex["liscr_usd_"+year] = liscr_usd
    trade_ex["direct_usd_"+year] = direct_usd
    
print("Inventory voyages table used for the following vessel types: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))
print("Global voyages table used for the following vessel types found: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_global_voys).unique()))
print("Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_ex.sea_dist_nm.min(), 1),
    round(trade_ex.sea_dist_nm.max(), 1),
    round(trade_ex.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_ex.vol_kg_vtw / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_ex.fobvalue / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.fobvalue.sum() / 1000), 1)
))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.bau_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.asis_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.brazil_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.liscr_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Direct: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.direct_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.direct_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))

trade_ex_agg = trade_ex.groupby(["cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "fobvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_ex_agg["vol_kg_vtw_pct"] = round(100 * trade_ex_agg.vol_kg_vtw / trade_ex_agg.vol_kg_vtw.sum(), 1)
trade_ex_agg["fobvalue_pct"] = round(100 * trade_ex_agg.fobvalue / trade_ex_agg.fobvalue.sum(), 1)
print("Most prominent 'cost intensity match' types...\n\n{0}\n".format(
    trade_ex_agg[["cost_int_match_desc", "vol_kg_vtw_pct", "fobvalue_pct"]].head(5)
))
trade_ex_agg_2 = trade_ex.groupby(["fac_vessel_types_str", "cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "fobvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_ex_agg_2["vol_kg_vtw_pct"] = round(100 * trade_ex_agg_2.vol_kg_vtw / trade_ex_agg_2.vol_kg_vtw.sum(), 1)
trade_ex_agg_2["fobvalue_pct"] = round(100 * trade_ex_agg_2.fobvalue / trade_ex_agg_2.fobvalue.sum(), 1)
print("Most prominent 'facilitating vessel types' and their 'cost intensity match' types...")
display(trade_ex_agg_2[["fac_vessel_types_str", "cost_int_match_desc", "vol_kg_vtw_pct", "fobvalue_pct"]].head(5))

trade_ex_na = trade_ex[(trade_ex[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_ex.fac_vessel_types.str.len() == 0)]
trade_ex = trade_ex[(~trade_ex[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_ex.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n".format(trade_ex_na.primaryValue.sum(), round(100 * trade_ex_na.primaryValue.sum() / trade_ex.primaryValue.sum(), 1)))

trade_ex.to_csv(output_dir + "{0}/impact_ex_{1}.csv".format(iso_country, version), index=False)
trade_ex.head(2)

Processing game year 2026
Processing game year 2029
Processing game year 2032
Processing game year 2035
Processing game year 2038
Processing game year 2041
Processing game year 2044
Processing game year 2047
Processing game year 2050
Inventory voyages table used for the following vessel types: 

[]

Global voyages table used for the following vessel types found: 

[]

Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: 

['2009 | Fruit or nut juices ' '2106 | Food preparations no'
 '3402 | Organic surface-acti' '9703 | Sculptures and statu'
 '4407 | Wood sawn or chipped' '4412 | Plywood, veneered pa'
 '2202 | Waters, including mi' "6202 | Coats; women's or gi"
 '9701 | Paintings, drawings ' '6117 | Clothing accessories'
 '1509 | Olive oil and its fr' '1510 | Vegetable oils; oils'
 '4707 | Waste and scrap of p' '1905 | Bread, pastry, cakes'
 '6704 | Wigs, false beards, ' '0910 | Ginger, saffron, tum'
 '7326 | Iron or steel; artic' '

,fac_vessel_types_str,cost_int_match_desc,vol_kg_vtw_pct,fobvalue_pct
0,['Bulk carrier'],1. Primary Int. Inv. with Vess Type,66.1,48.7
1,['Oil tanker'],1. Primary Int. Inv. with Vess Type,13.0,23.0
2,['Container'],1. Primary Int. Inv. with Vess Type,12.0,20.9
3,"['Container', 'General cargo']",1. Primary Int. Inv. with Vess Type,7.0,0.3
4,['all'],3. Primary Int. Inv. without Vess Type,1.3,5.3


Rows with NA: $0 (0.0%)



,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50
0,0,388,Jamaica,842,USA,000,World,Domestic Export,Sea,271019 | Petroleum oils and o,271019,kg,5.044147e+08,5.044147e+08,NaN,376375364,376375364,USA | 271019 | Petroleum oils and o,27,"27 | Mineral fuels, miner","USA | 27 | Mineral fuels, miner",2710,2710 | Petroleum oils and o,USA | 2710 | Petroleum oils and o,27.0,2710.0,271019.0,Petroleum oils & oils obtained from bituminous...,"Petroleum, Chemical and Non-Metallic Mineral P...",2320.0,Manufacture of refined petroleum products,3346.0,Petroleum oils & oils obtained from bituminous...,32.0,"P_C - Petroleum, coal products",333.0,Petroleum oils and oils obtained from bitumino...,0.6461,1.5476,both,1.340191,5.824785e+08,5.044147e+08,[Oil tanker],['Oil tanker'],1948.0,1210.430708,7.050499e+08,3 - Reporter and HS4 code,5.47,31,20587732.41,3.969631e+08,376375364,1. Primary Int. Inv. with Vess Type,1.804042e+06,1.800955e+06,1.801278e+06,1.801278e+06,1.802653e+06,1.802653e+06,1.874291e+06,3.054804e+06,1.990080e+06,1.878020e+06,1.885778e+06,1.949834e+06,1.844477e+06,3.031383e+06,2.123424e+06,1.966441e+06,1.935961e+06,2.007082e+06,1.813565e+06,3.344913e+06,2.517170e+06,2.319680e+06,2.169893e+06,2.080611e+06,1.783989e+06,3.628185e+06,2.961053e+06,2.795136e+06,2.265690e+06,2.210997e+06,1.753369e+06,3.914882e+06,3.548358e+06,3.668642e+06,2.405427e+06,2.316801e+06,1.723144e+06,3.473508e+06,3.551059e+06,3.795228e+06,3.090141e+06,2.354265e+06,1.694964e+06,3.303201e+06,3.260098e+06,3.725137e+06,3.005309e+06,2.375674e+06,1.666229e+06,2.886342e+06,3.005724e+06,3.331412e+06,2.907974e+06,2.388976e+06
1,1,388,Jamaica,842,USA,000,World,Domestic Export,Sea,330130 | Resinoids,330130,kg,1.707735e+05,1.707735e+05,NaN,599812,599812,USA | 330130 | Resinoids,33,33 | Essential oils and r,USA | 33 | Essential oils and r,3301,3301 | Oils; essential (con,USA | 3301 | Oils; essential (con,33.0,3301.0,330130.0,Resinoids,"Petroleum, Chemical and Non-Metallic Mineral P...",2429.0,Manufacture of other chemical products n.e.c.,55133.0,Resinoids,33.0,"CRP - Chemical, rubber, plastic products",3541.0,"Essential oils and concentrates, aqueous disti...",22.5983,0.0443,both,0.284712,2.657167e+04,1.707735e+05,"[Bulk carrier, Container]","['Bulk carrier', 'Container']",1948.0,1210.430708,3.216317e+04,3 - Reporter and HS4 code,2.78,12,16674.77,6.164868e+05,599812,1. Primary Int. Inv. with Vess Type,1.077974e+02,1.075331e+02,1.075581e+02,1.075581e+02,1.076956e+02,1.076956e+02,1.127847e+02,2.005156e+02,1.225009e+02,1.145633e+02,1.149563e+02,1.182596e+02,1.104231e+02,1.948171e+02,1.288518e+02,1.192240e+02,1.165708e+02,1.219354e+02,1.083523e+02,2.152488e+02,1.583314e+02,1.409498e+02,1.307360e+02,1.270144e+02,1.063173e+02,2.286796e+02,1.846320e+02,1.727000e+02,1.38584

## 3.4 Create Results Tables

### 3.4.1 Total Costs per Scenario

In [12]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_ex["bau_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["levy_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["asis_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["brazil_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["liscr_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["direct_usd_{0}".format(year)].sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% cif)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
    display(df)

    
print("\nTotal FEM costs by scenario (% fob)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_fob_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,36136137.08,35577725.86,35083113.63,34598333.10,34121030.39,33635221.69,33177891.18,32711615.90
1,Levy,56711739.45,54992403.16,60342587.79,66168863.73,68292685.17,63745284.77,59262239.47,55564948.91
2,'as-is',38021435.20,39396537.26,46990136.17,53486731.15,65021129.56,62273289.93,60473335.36,56636374.17
3,Brazil,36049978.60,37577295.49,43447669.18,50798986.66,67346975.24,66004103.52,65826775.23,64128788.45
4,LISCR,36158848.92,37068064.16,41094044.84,43010660.96,47236153.67,54574397.18,53174614.27,54434672.10
5,Direct,37383188.36,38233208.94,39404688.04,41517329.70,43066047.57,43620434.66,43969117.50,44181499.41



Total FEM costs by scenario (% ciffob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,34.14,33.62,33.15,32.69,32.24,31.78,31.35,30.91
1,Levy,53.58,51.96,57.01,62.52,64.53,60.23,55.99,52.50
2,'as-is',35.92,37.22,44.40,50.54,61.43,58.84,57.14,53.51
3,Brazil,34.06,35.50,41.05,48.00,63.63,62.36,62.20,60.59
4,LISCR,34.16,35.02,38.83,40.64,44.63,51.56,50.24,51.43
5,Direct,35.32,36.12,37.23,39.23,40.69,41.21,41.54,41.74



Total FEM costs by scenario (% cif)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,2.03,1.99,1.97,1.94,1.91,1.89,1.86,1.83
1,Levy,3.18,3.08,3.38,3.71,3.83,3.57,3.32,3.12
2,'as-is',2.13,2.21,2.63,3.00,3.65,3.49,3.39,3.18
3,Brazil,2.02,2.11,2.44,2.85,3.78,3.70,3.69,3.60
4,LISCR,2.03,2.08,2.30,2.41,2.65,3.06,2.98,3.05
5,Direct,2.10,2.14,2.21,2.33,2.41,2.45,2.46,2.48



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,2.15,2.12,2.09,2.06,2.03,2.00,1.98,1.95
1,Levy,3.38,3.28,3.60,3.94,4.07,3.80,3.53,3.31
2,'as-is',2.27,2.35,2.80,3.19,3.88,3.71,3.60,3.38
3,Brazil,2.15,2.24,2.59,3.03,4.01,3.93,3.92,3.82
4,LISCR,2.15,2.21,2.45,2.56,2.82,3.25,3.17,3.24
5,Direct,2.23,2.28,2.35,2.47,2.57,2.60,2.62,2.63



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.164,0.162,0.159,0.157,0.155,0.153,0.151,0.149
1,Levy,0.258,0.250,0.274,0.301,0.310,0.290,0.269,0.252
2,'as-is',0.173,0.179,0.213,0.243,0.295,0.283,0.275,0.257
3,Brazil,0.164,0.171,0.197,0.231,0.306,0.300,0.299,0.291
4,LISCR,0.164,0.168,0.187,0.195,0.215,0.248,0.242,0.247
5,Direct,0.170,0.174,0.179,0.189,0.196,0.198,0.200,0.201


### 3.4.2 Costs per Scenario relative to BAU

In [13]:
print("\nTotal FEM costs by scenario (US$)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_usd_bau_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal change in FEM costs by scenario (% bau)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_bau_{1}.csv".format(iso_country, version), index=False)
display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% cif)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_cif_{1}.csv".format(iso_country, version), index=False)
    display(df)


print("\nTotal FEM costs by scenario (% fob)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_fob_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,20575602.37,19414677.30,25259474.15,31570530.63,34171654.78,30110063.08,26084348.29,22853333.01
1,'as-is',1885298.12,3818811.40,11907022.54,18888398.05,30900099.16,28638068.23,27295444.18,23924758.27
2,Brazil,-86158.48,1999569.63,8364555.55,16200653.56,33225944.85,32368881.83,32648884.05,31417172.55
3,LISCR,22711.84,1490338.31,6010931.21,8412327.85,13115123.27,20939175.49,19996723.09,21723056.20
4,Direct,1247051.28,2655483.08,4321574.40,6918996.59,8945017.18,9985212.96,10791226.32,11469883.51



Total change in FEM costs by scenario (% bau)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,56.94,54.57,72.00,91.25,100.15,89.52,78.62,69.86
1,'as-is',5.22,10.73,33.94,54.59,90.56,85.14,82.27,73.14
2,Brazil,-0.24,5.62,23.84,46.82,97.38,96.24,98.41,96.04
3,LISCR,0.06,4.19,17.13,24.31,38.44,62.25,60.27,66.41
4,Direct,3.45,7.46,12.32,20.00,26.22,29.69,32.53,35.06



Total FEM costs by scenario (% ciffob)...



...

,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.15,1.09,1.42,1.77,1.92,1.69,1.46,1.28
1,'as-is',0.11,0.21,0.67,1.06,1.73,1.61,1.53,1.34
2,Brazil,-0.00,0.11,0.47,0.91,1.86,1.81,1.83,1.76
3,LISCR,0.00,0.08,0.34,0.47,0.74,1.17,1.12,1.22
4,Direct,0.07,0.15,0.24,0.39,0.50,0.56,0.60,0.64



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.23,1.16,1.51,1.88,2.04,1.79,1.55,1.36
1,'as-is',0.11,0.23,0.71,1.13,1.84,1.71,1.63,1.43
2,Brazil,-0.01,0.12,0.50,0.97,1.98,1.93,1.95,1.87
3,LISCR,0.00,0.09,0.36,0.50,0.78,1.25,1.19,1.29
4,Direct,0.07,0.16,0.26,0.41,0.53,0.60,0.64,0.68



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.093,0.088,0.115,0.143,0.155,0.137,0.119,0.104
1,'as-is',0.009,0.017,0.054,0.086,0.140,0.130,0.124,0.109
2,Brazil,-0.000,0.009,0.038,0.074,0.151,0.147,0.148,0.143
3,LISCR,0.000,0.007,0.027,0.038,0.060,0.095,0.091,0.099
4,Direct,0.006,0.012,0.020,0.031,0.041,0.045,0.049,0.052
